# CoLLM Injection Layer Experiment

**ML-1M 時間切分（對應原始論文）：**

| Split | 月份索引 | 對應時間 | 樣本數 |
|-------|---------|---------|-------|
| 歷史期（不生樣本）| 0–13 | 2000/04–2001/05 | — |
| Train | 14–23 | 2001/06–2002/03 | ~33,891 |
| Valid | 24–28 | 2002/04–2002/08 | ~10,401 |
| Test  | 29–33 | 2002/09–2003/01 | ~7,331 |

Label：rating > 3 → y=1，其餘 → y=0（不隨機採負樣本）

In [ ]:
# Cell 1: 確認 GPU
import subprocess, torch

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU:', result.stdout.strip())
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram_gb:.1f} GB')
    USE_8BIT = vram_gb < 20
else:
    USE_8BIT = False

print(f'load_in_8bit: {USE_8BIT}')

In [ ]:
# Cell 2: 掛載 Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

BASE     = '/content/drive/MyDrive/collm_experiment'
DATA_DIR = f'{BASE}/data/ml-1m'
CKPT_DIR = f'{BASE}/checkpoints'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print('Base directory:', BASE)

In [ ]:
# Cell 3: Clone repo 並安裝套件
import os

REPO_DIR = '/content/collm-refactor'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/hy1107/collm-refactor.git {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

os.system(f'pip install -q -e {REPO_DIR}')
os.system('pip install -q bitsandbytes accelerate')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Cell 4: 下載 ML-1M 資料集
import os

if not os.path.exists(f'{DATA_DIR}/ratings.dat'):
    os.system('wget -q -O /tmp/ml-1m.zip https://files.grouplens.org/datasets/movielens/ml-1m.zip')
    os.system('unzip -q /tmp/ml-1m.zip -d /tmp/')
    os.system(f'cp /tmp/ml-1m/ratings.dat {DATA_DIR}/')
    os.system(f'cp /tmp/ml-1m/movies.dat  {DATA_DIR}/')
    os.system(f'cp /tmp/ml-1m/users.dat   {DATA_DIR}/')
    print('ML-1M 下載完成')
else:
    print('ML-1M 已存在，跳過下載')

os.system(f'ls -lh {DATA_DIR}/')

In [ ]:
# Cell 5: 預處理資料（對應論文精確月份切分）
import os, pandas as pd

if not os.path.exists(f'{DATA_DIR}/train.pkl'):
    os.system(
        f'python scripts/preprocess_ml1m.py'
        f' --data_dir   {DATA_DIR}'
        f' --output_dir {DATA_DIR}'
    )
else:
    print('預處理資料已存在，跳過（如需重跑請先刪除 train.pkl）')

for split in ['train', 'valid', 'test']:
    df = pd.read_pickle(f'{DATA_DIR}/{split}.pkl')
    pos = (df['label'] == 1).sum()
    neg = (df['label'] == 0).sum()
    print(f'{split}: {len(df):>7} 筆（pos={pos}, neg={neg}）')

In [ ]:
# Cell 6: 訓練 Rec Encoder (SASRec)
import os

REC_CKPT = f'{CKPT_DIR}/sasrec_ml1m.pth'

if not os.path.exists(REC_CKPT):
    os.system(
        f'python scripts/train_rec_encoder.py'
        f' --data_path {DATA_DIR}/train'
        f' --encoder_type sasrec'
        f' --embedding_dim 64'
        f' --n_layers 2'
        f' --n_heads 2'
        f' --max_seq_len 50'
        f' --epochs 30'
        f' --lr 1e-3'
        f' --batch_size 256'
        f' --save_path {REC_CKPT}'
    )
else:
    print(f'Rec encoder 已存在：{REC_CKPT}')

In [ ]:
# Cell 7: 更新 YAML 設定（路徑 + 8bit）
import yaml

with open('configs/stage1_movielens.yaml') as f:
    cfg1 = yaml.safe_load(f)
cfg1['data']['data_path']        = DATA_DIR
cfg1['backbone']['load_in_8bit'] = bool(USE_8BIT)
with open('configs/stage1_movielens.yaml', 'w') as f:
    yaml.dump(cfg1, f, allow_unicode=True)

with open('configs/stage2_movielens.yaml') as f:
    cfg2 = yaml.safe_load(f)
cfg2['data']['data_path']        = DATA_DIR
cfg2['rec']['checkpoint_path']   = REC_CKPT
cfg2['backbone']['load_in_8bit'] = bool(USE_8BIT)
with open('configs/stage2_movielens.yaml', 'w') as f:
    yaml.dump(cfg2, f, allow_unicode=True)

print('Stage 1 config:')
print(open('configs/stage1_movielens.yaml').read())
print('Stage 2 config:')
print(open('configs/stage2_movielens.yaml').read())

In [ ]:
# Cell 8: Stage 1 - LoRA fine-tune
# T4 上約需 2-3 小時，A100 約 40 分鐘
import os

STAGE1_CKPT = f'{CKPT_DIR}/stage1'
BS = 2 if USE_8BIT else 4

if not os.path.exists(f'{STAGE1_CKPT}/adapter_config.json'):
    os.system(
        f'python scripts/train_stage1.py'
        f' --config configs/stage1_movielens.yaml'
        f' --output_dir {STAGE1_CKPT}'
        f' --num_train_epochs 3'
        f' --per_device_train_batch_size {BS}'
        f' --learning_rate 2e-4'
    )
else:
    print(f'Stage 1 checkpoint 已存在：{STAGE1_CKPT}')

In [ ]:
# Cell 9: Stage 2 - 單一 layer 快速測試（layer=0，確認流程）
import os

BS = 2 if USE_8BIT else 4
TEST_LAYER = 0

os.system(
    f'python scripts/train_stage2.py'
    f' --config configs/stage2_movielens.yaml'
    f' --stage1_checkpoint {STAGE1_CKPT}'
    f' --output_dir {CKPT_DIR}/stage2_layer{TEST_LAYER}'
    f' --injection_layer {TEST_LAYER}'
    f' --num_train_epochs 1'
    f' --per_device_train_batch_size {BS}'
    f' --learning_rate 1e-4'
)

In [ ]:
# Cell 10: Stage 2 - 多 layer sweep
import os

LAYERS = [0, 3, 10, 16, 20, 23]
BS = 2 if USE_8BIT else 4

for layer in LAYERS:
    out_dir = f'{CKPT_DIR}/stage2_layer{layer}'
    print(f'\n====== injection_layer={layer} ======')
    os.system(
        f'python scripts/train_stage2.py'
        f' --config configs/stage2_movielens.yaml'
        f' --stage1_checkpoint {STAGE1_CKPT}'
        f' --output_dir {out_dir}'
        f' --injection_layer {layer}'
        f' --num_train_epochs 2'
        f' --per_device_train_batch_size {BS}'
        f' --learning_rate 1e-4'
    )
    print(f'<<< layer {layer} 完成')

In [ ]:
# Cell 11: 比較各 layer 的 eval metrics
import os, json
import pandas as pd

LAYERS = [0, 3, 10, 16, 20, 23]
rows = []

for layer in LAYERS:
    state_path = f'{CKPT_DIR}/stage2_layer{layer}/trainer_state.json'
    if not os.path.exists(state_path):
        continue
    with open(state_path) as f:
        state = json.load(f)
    eval_logs = [x for x in state.get('log_history', []) if 'eval_auc' in x]
    if eval_logs:
        last = eval_logs[-1]
        rows.append({
            'injection_layer': layer,
            'auc':     round(last.get('eval_auc',     0), 4),
            'hr@10':   round(last.get('eval_hr@10',   0), 4),
            'ndcg@10': round(last.get('eval_ndcg@10', 0), 4),
        })

if rows:
    df = pd.DataFrame(rows).set_index('injection_layer')
    print(df.to_string())
    df.to_csv(f'{CKPT_DIR}/sweep_results.csv')
    print(f'\n結果已存至 {CKPT_DIR}/sweep_results.csv')
else:
    print('尚無 eval 結果，請先完成 Cell 10')